# Loop-TNR Colab

Runs directly against the `.py` files in this repo. Edit `REPO` below to point at the repo directory.

In [ ]:
import os, sys, pathlib
REPO = '/content/drive/MyDrive/Loop-TNR-sandbox'  # edit if needed
os.chdir(REPO)
sys.path.insert(0, REPO)
assert pathlib.Path('fast_engine.py').is_file(), f'wrong REPO: {REPO}'
assert pathlib.Path('utils.py').is_file(), 'utils.py missing from repo'
from utils import dict_diff, dict_squared_norm, dict_frobenius_norm, dict_to_vec, vec_to_dict

In [ ]:
!pip install -q opt_einsum

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
print(jax.default_backend(), jax.devices())

## Smoke: chi=2 RG step

In [ ]:
from fast_engine import LoopTNREngine
e = LoopTNREngine(Q=2, chi=2); e.set_beta(e.beta_c)
r, eps2 = e.rg_step(maxiter=200)
print(f'eps2={eps2:.3e} nit={r.nit}')

## Dict-tensor utilities spot-check

In [ ]:
from loss_function_and_gradient import setup_potts, compute_loss_from_state
state = setup_potts(Q_potts=2, verbose=False)
diff = dict_diff(state['U_tensor'], state['V_tensor'])
print(f"loss via dict_squared_norm = {dict_squared_norm(diff):.3e}")
print(f"||diff||_F via dict_frobenius_norm = {dict_frobenius_norm(diff):.3e}")
print(f"compute_loss_from_state       = {compute_loss_from_state(state):.3e}")

## Validation (fast: Q=2, chi in {2,4})

In [ ]:
!python validate.py --fast --output_dir ./validation_out

## RG flow Q=2 chi=4, 4 steps at beta_c

In [ ]:
e = LoopTNREngine(Q=2, chi=4); e.set_beta(e.beta_c)
e.rg_flow(n_steps=4, maxiter=200)

## Bidirectional beta sweep (locates beta_c)

In [ ]:
import numpy as np
e = LoopTNREngine(Q=2, chi=4); e.set_beta(e.beta_c)
betas = np.linspace(0.85*e.beta_c, 1.15*e.beta_c, 7)
res = e.beta_sweep_bidirectional(betas, num_rg_steps=3, maxiter=150, verbose=False)
i = int(np.argmax(res['avg_total_iters']))
print(f"est beta_c = {res['betas'][i]:.5f}  exact = {e.beta_c:.5f}")